In [2]:
import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    precision_score,
    accuracy_score,
    roc_auc_score,
    balanced_accuracy_score
)

import joblib
import os
import warnings

warnings.filterwarnings("ignore")

# ==============================
# 1. LOAD DATA
# ==============================

X = sparse.load_npz(
    "E:/LogAnomalyDetector/data/processed/X_features.npz"
)

metadata = pd.read_csv(
    "E:/LogAnomalyDetector/data/processed/y_labels.csv"
)

y = metadata["true_label"].values

source_dataset = (
    metadata["source_dataset"]
    .values
)

print("Dataset Shape:", X.shape)

print("\nLabel Distribution:")

print(
    metadata["true_label"]
    .value_counts()
)

print("\nDataset Distribution:")

print(
    metadata["source_dataset"]
    .value_counts()
)

print("\nDataset-wise Label Distribution:")

print(
    pd.crosstab(
        metadata["source_dataset"],
        metadata["true_label"]
    )
)

# ==============================
# 2. INDEX TRACKING
# ==============================

indices = np.arange(len(y))

# ==============================
# 3. STRATIFIED TRAIN-TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X,
    y,
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain Shape:", X_train.shape)

print("Test Shape:", X_test.shape)

# Save test indices
os.makedirs(
    "E:/LogAnomalyDetector/data/processed",
    exist_ok=True
)

np.save(
    "E:/LogAnomalyDetector/data/processed/test_indices.npy",
    idx_test
)

# ==============================
# 4. INITIALIZE MODELS
# ==============================

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

svm_base = LinearSVC(
    class_weight="balanced",
    random_state=42
)

# ==============================
# 5. CALIBRATED SVM
# ==============================

svm = CalibratedClassifierCV(
    estimator=svm_base,
    method="sigmoid",
    cv=3
)

# ==============================
# 6. TRAIN MODELS
# ==============================

print("\nTraining Models...")

rf.fit(
    X_train,
    y_train
)

lr.fit(
    X_train,
    y_train
)

svm.fit(
    X_train,
    y_train
)

print("\n✅ All Models Trained Successfully")

# ==============================
# 7. RANDOM FOREST
#    THRESHOLD TUNING
# ==============================

print("\n🌲 Random Forest Threshold Optimization")

y_prob_rf = (
    rf.predict_proba(X_test)[:, 1]
)

thresholds = np.arange(
    0.10,
    0.90,
    0.05
)

best_t = 0.50

best_f1 = 0

print("\n🔍 Threshold Tuning Results:")

for t in thresholds:

    preds = (
        y_prob_rf >= t
    ).astype(int)

    acc = accuracy_score(
        y_test,
        preds
    )

    f1 = f1_score(
        y_test,
        preds
    )

    rec = recall_score(
        y_test,
        preds
    )

    prec = precision_score(
        y_test,
        preds
    )

    bal_acc = balanced_accuracy_score(
        y_test,
        preds
    )

    print(
        f"t={t:.2f} | "
        f"ACC={acc:.3f} | "
        f"BAL_ACC={bal_acc:.3f} | "
        f"F1={f1:.3f} | "
        f"Recall={rec:.3f} | "
        f"Precision={prec:.3f}"
    )

    if f1 > best_f1:

        best_f1 = f1

        best_t = t

print(
    f"\n✅ Best Threshold Selected: "
    f"{best_t:.2f}"
)

# ==============================
# 8. RANDOM FOREST EVALUATION
# ==============================

y_pred_rf = (
    y_prob_rf >= best_t
).astype(int)

rf_auc = roc_auc_score(
    y_test,
    y_prob_rf
)

print("\n📊 Random Forest Results")

print(
    f"\nROC-AUC Score: "
    f"{rf_auc:.4f}"
)

print("\nClassification Report (RF):")

print(
    classification_report(
        y_test,
        y_pred_rf,
        zero_division=0
    )
)

print("\nConfusion Matrix (RF):")

print(
    confusion_matrix(
        y_test,
        y_pred_rf
    )
)

# ==============================
# 9. LOGISTIC REGRESSION
# ==============================

print("\n📈 Logistic Regression")

y_prob_lr = (
    lr.predict_proba(X_test)[:, 1]
)

y_pred_lr = lr.predict(
    X_test
)

lr_auc = roc_auc_score(
    y_test,
    y_prob_lr
)

print(
    f"\nROC-AUC Score: "
    f"{lr_auc:.4f}"
)

print("\nClassification Report (LR):")

print(
    classification_report(
        y_test,
        y_pred_lr,
        zero_division=0
    )
)

print("\nConfusion Matrix (LR):")

print(
    confusion_matrix(
        y_test,
        y_pred_lr
    )
)

# ==============================
# 10. CALIBRATED SVM
# ==============================

print("\n⚡ Calibrated Linear SVM")

y_prob_svm = (
    svm.predict_proba(X_test)[:, 1]
)

y_pred_svm = svm.predict(
    X_test
)

svm_auc = roc_auc_score(
    y_test,
    y_prob_svm
)

print(
    f"\nROC-AUC Score: "
    f"{svm_auc:.4f}"
)

print("\nClassification Report (SVM):")

print(
    classification_report(
        y_test,
        y_pred_svm,
        zero_division=0
    )
)

print("\nConfusion Matrix (SVM):")

print(
    confusion_matrix(
        y_test,
        y_pred_svm
    )
)

# ==============================
# 11. FEATURE IMPORTANCE
# ==============================

importances = (
    rf.feature_importances_
)

top_features = sorted(
    importances,
    reverse=True
)[:10]

print(
    "\nTop 10 Important Features (RF):"
)

print(top_features)

# ==============================
# 12. SAVE BASE MODELS
# ==============================

os.makedirs(
    "E:/LogAnomalyDetector/models",
    exist_ok=True
)

joblib.dump(
    rf,
    "E:/LogAnomalyDetector/models/random_forest_model.joblib"
)

joblib.dump(
    lr,
    "E:/LogAnomalyDetector/models/logistic_regression_model.joblib"
)

joblib.dump(
    svm,
    "E:/LogAnomalyDetector/models/calibrated_linear_svm_model.joblib"
)

joblib.dump(
    best_t,
    "E:/LogAnomalyDetector/models/best_rf_threshold.joblib"
)

print(
    "\n✅ Base Models Saved Successfully"
)

# ==============================
# 13. CROSS-DATASET ANALYSIS
# ==============================

hdfs_mask_np = (
    metadata["source_dataset"]
    .values == "hdfs"
)

bgl_mask_np = (
    metadata["source_dataset"]
    .values == "bgl"
)

print("\nHDFS Samples:")
print(hdfs_mask_np.sum())

print("\nBGL Samples:")
print(bgl_mask_np.sum())

# ==============================
# 14. TRAIN HDFS -> TEST BGL
# ==============================

X_train_cross = X[hdfs_mask_np]

y_train_cross = y[hdfs_mask_np]

X_test_cross = X[bgl_mask_np]

y_test_cross = y[bgl_mask_np]

print("\nCross-Dataset Shapes:")

print("Train:", X_train_cross.shape)

print("Test:", X_test_cross.shape)

cross_rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\nTraining RF on HDFS only...")

cross_rf.fit(
    X_train_cross,
    y_train_cross
)

print("\nTesting RF on BGL only...")

y_pred_cross = cross_rf.predict(
    X_test_cross
)

print("\nCross-Dataset Classification Report:")

print(
    classification_report(
        y_test_cross,
        y_pred_cross,
        zero_division=0
    )
)

print("\nCross-Dataset Confusion Matrix:")

print(
    confusion_matrix(
        y_test_cross,
        y_pred_cross
    )
)

# ==============================
# 15. TRAIN BGL -> TEST HDFS
# ==============================

X_train_bgl = X[bgl_mask_np]

y_train_bgl = y[bgl_mask_np]

X_test_hdfs = X[hdfs_mask_np]

y_test_hdfs = y[hdfs_mask_np]

print("\nTraining RF on BGL only...")

reverse_rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

reverse_rf.fit(
    X_train_bgl,
    y_train_bgl
)

print("\nTesting RF on HDFS only...")

y_pred_reverse = reverse_rf.predict(
    X_test_hdfs
)

print("\nReverse Cross-Dataset Report:")

print(
    classification_report(
        y_test_hdfs,
        y_pred_reverse,
        zero_division=0
    )
)

print("\nReverse Cross-Dataset Confusion Matrix:")

print(
    confusion_matrix(
        y_test_hdfs,
        y_pred_reverse
    )
)

# ==============================
# 16. RANDOM FOREST TUNING
# ==============================

print("\n🔧 Starting Random Forest Hyperparameter Tuning...")

rf_params = {

    "n_estimators": [
        100,
        200,
        300
    ],

    "max_depth": [
        10,
        20,
        30,
        None
    ],

    "min_samples_split": [
        2,
        5,
        10
    ],

    "min_samples_leaf": [
        1,
        2,
        4
    ],

    "max_features": [
        "sqrt",
        "log2"
    ]
}

cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

rf_tuner = RandomizedSearchCV(

    estimator=RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    param_distributions=rf_params,

    n_iter=10,

    scoring="f1",

    cv=cv_strategy,

    verbose=2,

    random_state=42,

    n_jobs=-1
)

rf_tuner.fit(
    X_train,
    y_train
)

best_rf = rf_tuner.best_estimator_

print("\n✅ Best RF Parameters:")

print(
    rf_tuner.best_params_
)

# ==============================
# 17. TUNED RF EVALUATION
# ==============================

y_prob_tuned = (
    best_rf.predict_proba(X_test)[:, 1]
)

y_pred_tuned = (
    y_prob_tuned >= best_t
).astype(int)

tuned_auc = roc_auc_score(
    y_test,
    y_prob_tuned
)

print(
    f"\n📊 Tuned RF ROC-AUC: "
    f"{tuned_auc:.4f}"
)

print("\n📊 Tuned RF Classification Report:")

print(
    classification_report(
        y_test,
        y_pred_tuned,
        zero_division=0
    )
)

print("\n📊 Tuned RF Confusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred_tuned
    )
)

# ==============================
# 18. SAVE TUNED MODEL
# ==============================

joblib.dump(
    best_rf,
    "E:/LogAnomalyDetector/models/tuned_random_forest_model.joblib"
)

print(
    "\n✅ Tuned Random Forest Saved"
)

# ==============================
# 19. TUNED FEATURE IMPORTANCE
# ==============================

tuned_importances = (
    best_rf.feature_importances_
)

top_tuned = sorted(
    tuned_importances,
    reverse=True
)[:10]

print(
    "\nTop 10 Tuned RF Features:"
)

print(top_tuned)

# ==============================
# 20. MODEL COMPARISON SUMMARY
# ==============================

summary_df = pd.DataFrame({

    "Model": [
        "Random Forest",
        "Logistic Regression",
        "Calibrated SVM"
    ],

    "ROC_AUC": [
        rf_auc,
        lr_auc,
        svm_auc
    ]
})

print("\n📊 Model Comparison Summary:")

print(summary_df)

# ==============================
# 21. FINAL SUMMARY
# ==============================

print("\n===================================")
print(" FINAL TRAINING PIPELINE COMPLETE ")
print("===================================")

print("\nCompleted Sections:")
print("✔ Multi-Model Training")
print("✔ Behavioral Feature Learning")
print("✔ Threshold Optimization")
print("✔ Probability Calibration")
print("✔ Cross-Dataset Validation")
print("✔ Dataset Bias Analysis")
print("✔ Hyperparameter Tuning")
print("✔ Tuned Model Evaluation")
print("✔ ROC-AUC Evaluation")
print("✔ Model Persistence")

Dataset Shape: (4000, 1330)

Label Distribution:
true_label
0    2074
1    1926
Name: count, dtype: int64

Dataset Distribution:
source_dataset
hdfs    2000
bgl     2000
Name: count, dtype: int64

Dataset-wise Label Distribution:
true_label         0     1
source_dataset            
bgl              143  1857
hdfs            1931    69

Train Shape: (3200, 1330)
Test Shape: (800, 1330)

Training Models...

✅ All Models Trained Successfully

🌲 Random Forest Threshold Optimization

🔍 Threshold Tuning Results:
t=0.10 | ACC=0.971 | BAL_ACC=0.971 | F1=0.970 | Recall=0.969 | Precision=0.971
t=0.15 | ACC=0.973 | BAL_ACC=0.972 | F1=0.971 | Recall=0.969 | Precision=0.974
t=0.20 | ACC=0.973 | BAL_ACC=0.972 | F1=0.971 | Recall=0.969 | Precision=0.974
t=0.25 | ACC=0.974 | BAL_ACC=0.974 | F1=0.973 | Recall=0.969 | Precision=0.976
t=0.30 | ACC=0.974 | BAL_ACC=0.974 | F1=0.973 | Recall=0.969 | Precision=0.976
t=0.35 | ACC=0.978 | BAL_ACC=0.977 | F1=0.976 | Recall=0.969 | Precision=0.984
t=0.40 | ACC=